# 03. Coherent States, Displacement Operators & Poissonian Statistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_1_states_of_light/03_coherent_states_and_displacement.ipynb)

* **Module:** 1 — Single-Mode Quantum States of Light
* **Difficulty:** Intermediate
* **Estimated Time:** 45 minutes
* **Prerequisites:** Notebook 1.1 (Fock States & Cutoff Convergence), Notebook 0.5 (Time Evolution & Wave Packets).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Define and construct Coherent States** $|\alpha\rangle$ in QuTiP using three equivalent physical definitions (eigenstate of $\hat{a}$, displaced vacuum, and minimum uncertainty state).
2. **Implement the Glauber Displacement Operator** $\hat{D}(\alpha) = \exp(\alpha\hat{a}^\dagger - \alpha^*\hat{a})$ and verify its composition law $\hat{D}(\alpha)\hat{D}(\beta) = \hat{D}(\alpha+\beta)e^{i\,\mathrm{Im}(\alpha\beta^*)}$.
3. **Analyze the Poissonian photon number distribution** $P(n) = e^{-|\alpha|^2}\frac{|\alpha|^{2n}}{n!}$ and verify that the number variance equals the mean ($\langle (\Delta n)^2 \rangle = \bar{n}$).
4. **Demonstrate minimum quadrature uncertainty** $\Delta X_1 \Delta X_2 = 1/4$ independent of amplitude $|\alpha|$, visualizing coherent states as displaced phase-space circles.
5. **Prove non-orthogonality and overcompleteness**: calculate the overlap $|\langle \beta | \alpha \rangle|^2 = \exp(-|\alpha - \beta|^2)$ and explain its role in continuous-variable quantum information.
6. **Simulate harmonic phase rotation**: prove that a coherent state evolves as $|\alpha(t)\rangle = |\alpha e^{-i\omega t}\rangle$, rotating clockwise on the phase plane at frequency $\omega$.


---
## 0. Setup & Dependencies

We will use **QuTiP (Quantum Toolbox in Python)** alongside NumPy, SciPy, and Matplotlib.


In [ ]:
# Install dependencies in Google Colab (uncomment if running in Colab)
# !pip install -q qutip numpy scipy matplotlib ipywidgets

import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
import math

# QuTiP import
try:
    import qutip as qt
    from qutip import basis, destroy, create, num, coherent, displace, qeye
    has_qutip = True
    print(f"QuTiP version {qt.__version__} initialized.")
except ImportError:
    has_qutip = False
    print("QuTiP not installed locally. Using internal NumPy fallback.")

# Configure clean, publication-ready plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})


---
## 1. Three Equivalent Definitions of Coherent States

Coherent states $|\alpha\rangle$ were discovered by Erwin Schrödinger in 1926 as non-spreading quantum wave packets and formulated systematically by Roy J. Glauber in 1963 to describe laser radiation. 

A single-mode coherent state can be defined in **three mathematically equivalent ways**:

### Definition 1: Right-Eigenstate of the Annihilation Operator
Because the annihilation operator $\hat{a}$ is non-Hermitian ($\hat{a}^\dagger \ne \hat{a}$), its eigenvalues are not restricted to real numbers; they can be any arbitrary complex number $\alpha \in \mathbb{C}$:

$$\hat{a}|\alpha\rangle = \alpha |\alpha\rangle, \quad \alpha = |\alpha| e^{i\theta} = x_1 + i x_2$$

The complex eigenvalue $\alpha$ directly encodes:
* Amplitude $|\alpha| = \sqrt{\bar{n}}$ (proportional to the classical electric field magnitude).
* Optical phase $\theta = \arg(\alpha)$.

### Definition 2: The Displaced Vacuum State
A coherent state is generated by applying the unitary **displacement operator** $\hat{D}(\alpha)$ to the electromagnetic vacuum $|0\rangle$:

$$|\alpha\rangle = \hat{D}(\alpha)|0\rangle \quad \text{where } \hat{D}(\alpha) \equiv \exp\left( \alpha\hat{a}^\dagger - \alpha^*\hat{a} \right)$$

### Definition 3: Minimum Uncertainty State with Equal Noise
A coherent state is a minimum uncertainty state:

$$\Delta X_1 = \Delta X_2 = \frac{1}{2} \implies \Delta X_1 \Delta X_2 = \frac{1}{4}$$

centered at non-zero mean quadrature displacements:
$$\langle \hat{X}_1 \rangle = \mathrm{Re}(\alpha), \quad \langle \hat{X}_2 \rangle = \mathrm{Im}(\alpha)$$


In [ ]:
def construct_coherent_state(N: int, alpha: complex):
    """
    Constructs a coherent state |alpha> in truncated Fock space of dimension N.
    """
    if has_qutip:
        return coherent(N, alpha)
    else:
        # NumPy fallback: |alpha> = exp(-|alpha|^2 / 2) * sum(alpha^n / sqrt(n!) |n>)
        coeffs = np.zeros(N, dtype=complex)
        for n in range(N):
            coeffs[n] = (alpha**n) / math.sqrt(float(math.factorial(n)))
        psi = np.exp(-0.5 * abs(alpha)**2) * coeffs
        # Normalize to handle truncation
        norm = np.linalg.norm(psi)
        return psi / norm

# Benchmark Definition 1: Verify that a |alpha> = alpha |alpha>
N_dim = 35
alpha_test = 2.0 + 1.5j  # |alpha|^2 = 4 + 2.25 = 6.25 mean photons

if has_qutip:
    psi_alpha = construct_coherent_state(N_dim, alpha_test)
    a_op = destroy(N_dim)
    
    # Calculate a * |alpha>
    lhs = a_op * psi_alpha
    rhs = alpha_test * psi_alpha
    
    diff_norm = (lhs - rhs).norm()
    print("=== Definition 1 Verification: a|alpha> = alpha|alpha> ===")
    print(f"Target eigenvalue alpha: {alpha_test}")
    print(f"Norm of difference || a|alpha> - alpha|alpha> ||: {diff_norm:.2e} (Exact eigenstate!)")
else:
    print(f"Target eigenvalue alpha: {alpha_test} (QuTiP verification skipped)")


---
## 2. The Displacement Operator $\hat{D}(\alpha)$ and its Algebra

The displacement operator $\hat{D}(\alpha) = \exp(\alpha\hat{a}^\dagger - \alpha^*\hat{a})$ is the single most important unitary transformation in quantum optics.

### 1. Disentangling Formula (BCH Factorization)
Using the Baker-Campbell-Hausdorff relation for operators whose commutator is a c-number ($[\hat{a}, \hat{a}^\dagger] = 1$):
$$e^{\hat{A} + \hat{B}} = e^{\hat{A}} e^{\hat{B}} e^{-\frac{1}{2}[\hat{A}, \hat{B}]}$$

Setting $\hat{A} = \alpha\hat{a}^\dagger$ and $\hat{B} = -\alpha^*\hat{a}$, with $[\alpha\hat{a}^\dagger, -\alpha^*\hat{a}] = |\alpha|^2$:

$$\hat{D}(\alpha) = e^{-|\alpha|^2 / 2} \exp\left( \alpha \hat{a}^\dagger \right) \exp\left( -\alpha^* \hat{a} \right)$$

Applying this form to the vacuum $|0\rangle$, since $e^{-\alpha^*\hat{a}}|0\rangle = |0\rangle$:

$$|\alpha\rangle = \hat{D}(\alpha)|0\rangle = e^{-|\alpha|^2 / 2} \sum_{n=0}^{\infty} \frac{\alpha^n}{\sqrt{n!}} |n\rangle$$

### 2. Group Composition Law
Displacing by $\beta$ and then by $\alpha$ is equivalent to a single displacement by $\alpha + \beta$, up to a geometric **Berry / Aharonov-Anandan phase factor**:

$$\hat{D}(\alpha)\hat{D}(\beta) = \hat{D}(\alpha + \beta) \exp\left( \frac{\alpha\beta^* - \alpha^*\beta}{2} \right) = \hat{D}(\alpha + \beta) e^{i\,\mathrm{Im}(\alpha\beta^*)}$$

### 3. Shift of Ladder Operators
Under displacement unitary similarity transformation:

$$\hat{D}^\dagger(\alpha)\hat{a}\hat{D}(\alpha) = \hat{a} + \alpha$$

$$\hat{D}^\dagger(\alpha)\hat{a}^\dagger\hat{D}(\alpha) = \hat{a}^\dagger + \alpha^*$$


In [ ]:
def test_displacement_composition(alpha: complex, beta: complex, N: int = 40):
    """
    Numerically verifies D(alpha) * D(beta) = D(alpha + beta) * exp(i * Im(alpha * conj(beta))).
    """
    if has_qutip:
        D_alpha = displace(N, alpha)
        D_beta = displace(N, beta)
        D_sum = displace(N, alpha + beta)
        
        # Left-hand side: D(alpha) * D(beta)
        LHS = D_alpha * D_beta
        
        # Right-hand side: D(alpha + beta) * exp(i * Im(alpha * beta*))
        phase = np.exp(1j * np.imag(alpha * np.conj(beta)))
        RHS = D_sum * phase
        
        operator_diff = (LHS - RHS).norm()
        print("=== Displacement Composition Law Verification ===")
        print(f"alpha = {alpha}, beta = {beta}")
        print(f"Geometric Phase: exp(i * Im(alpha*beta*)) = {phase:.4f}")
        print(f"Operator difference || D(alpha)D(beta) - D(alpha+beta)e^(i*phi) ||: {operator_diff:.2e}")
    else:
        print("QuTiP required for full operator composition test.")

test_displacement_composition(alpha=1.2 + 0.8j, beta=-0.5 + 1.1j, N=45)


---
## 3. Photon Number Statistics: The Poissonian Distribution

Expanding $|\alpha\rangle$ in the discrete Fock basis:

$$|\alpha\rangle = \sum_{n=0}^{\infty} c_n |n\rangle, \quad c_n = e^{-|\alpha|^2 / 2} \frac{\alpha^n}{\sqrt{n!}}$$

The probability of detecting exactly $n$ photons in the mode is given by Born's rule:

$$P(n) = |\langle n | \alpha \rangle|^2 = |c_n|^2 = e^{-|\alpha|^2} \frac{|\alpha|^{2n}}{n!} = e^{-\bar{n}} \frac{\bar{n}^n}{n!}$$

This is the classic **Poisson distribution** with mean parameter $\bar{n} = |\alpha|^2$.

### Key Properties of Poissonian Light
1. **Mean Photon Number:** $\langle \hat{n} \rangle = |\alpha|^2 = \bar{n}$
2. **Photon Number Variance:** 
   $$\langle (\Delta n)^2 \rangle = \langle \hat{n}^2 \rangle - \langle \hat{n} \rangle^2 = \bar{n}$$
   The variance equals the mean!
3. **The Fano Factor:**
   $$F \equiv \frac{\langle (\Delta n)^2 \rangle}{\langle \hat{n} \rangle} = \frac{\bar{n}}{\bar{n}} = 1.0$$
   * $F = 1$: **Poissonian light** (Coherent laser beam).
   * $F < 1$: **Sub-Poissonian light** (Non-classical Fock states, squeezed states).
   * $F > 1$: **Super-Poissonian light** (Thermal, chaotic light).
4. **Relative Intensity Fluctuations:**
   $$\frac{\Delta n}{\bar{n}} = \frac{\sqrt{\bar{n}}}{\bar{n}} = \frac{1}{\sqrt{\bar{n}}} \xrightarrow{\bar{n} \to \infty} 0$$
   For a macroscopic laser pointer ($\bar{n} \sim 10^{15}$ photons/sec), relative quantum fluctuations are on the order of $10^{-7.5} \approx 3 \times 10^{-8}$—completely imperceptible, perfectly matching classical electromagnetic waves!


In [ ]:
def plot_poisson_statistics():
    alpha_samples = [1.0, 2.0, 3.5, 5.0]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    axes_flat = axes.flatten()
    
    n_max = 40
    fock_axis = np.arange(n_max)
    
    for idx, alpha_val in enumerate(alpha_samples):
        n_bar = alpha_val**2
        ax = axes_flat[idx]
        
        # Analytical Poisson probabilities: P(n) = exp(-n_bar) * n_bar^n / n!
        poiss_probs = [np.exp(-n_bar) * (n_bar**n) / float(math.factorial(n)) for n in fock_axis]
        
        # Numerical expectation values
        mean_calc = np.sum(fock_axis * poiss_probs)
        mean_sq_calc = np.sum((fock_axis**2) * poiss_probs)
        var_calc = mean_sq_calc - mean_calc**2
        fano = var_calc / mean_calc
        
        ax.bar(fock_axis, poiss_probs, color="#1f77b4", alpha=0.75, width=0.65, edgecolor="black", lw=0.8)
        ax.axvline(mean_calc, color="#d62728", ls="--", lw=2, label=rf"Mean $\bar{{n}} = {mean_calc:.2f}$")
        ax.axvspan(mean_calc - np.sqrt(var_calc), mean_calc + np.sqrt(var_calc), color="#d62728", alpha=0.15,
                   label=rf"Spread $\pm \Delta n = \pm{np.sqrt(var_calc):.2f}$")
        
        ax.set_title(rf"Coherent State $|\alpha={alpha_val}\rangle$ ($\bar{{n}} = {n_bar:.1f}$, Fano $F = {fano:.2f}$)")
        ax.set_xlabel("Photon Number $n$")
        ax.set_ylabel("Probability $P(n)$")
        ax.set_xlim(-0.5, min(n_max, int(n_bar + 4*np.sqrt(n_bar) + 3)))
        ax.legend(loc="upper right")
        
    plt.tight_layout()
    plt.show()

plot_poisson_statistics()


> **Physical Insight:** For small amplitudes ($\alpha = 1.0$), the distribution peaks at $n=0$ and $n=1$. As the amplitude increases ($\alpha = 5.0, \bar{n} = 25$), the Poisson distribution becomes symmetric and approaches a continuous **Gaussian curve** with width $\Delta n = \sqrt{\bar{n}} = 5.0$, consistent with the central limit theorem.


---
## 4. Phase Space Geometry: Displaced Minimum Uncertainty Circles

Recall the optical quadrature operators:

$$\hat{X}_1 = \frac{\hat{a} + \hat{a}^\dagger}{2}, \quad \hat{X}_2 = \frac{\hat{a} - \hat{a}^\dagger}{2i}$$

Let us compute the expectation values and uncertainties for a coherent state $|\alpha\rangle$:

1. **Mean Quadratures:**
   $$\langle \alpha | \hat{X}_1 | \alpha \rangle = \frac{\alpha + \alpha^*}{2} = \mathrm{Re}(\alpha)$$
   $$\langle \alpha | \hat{X}_2 | \alpha \rangle = \frac{\alpha - \alpha^*}{2i} = \mathrm{Im}(\alpha)$$

2. **Quadrature Variances:**
   Using $\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$ and $[\hat{a}, \hat{a}^\dagger] = 1$:
   $$\langle \hat{X}_1^2 \rangle = \frac{1}{4}\langle \alpha | (\hat{a}^2 + \hat{a}^{\dagger 2} + 2\hat{a}^\dagger\hat{a} + 1) | \alpha \rangle = \mathrm{Re}(\alpha)^2 + \frac{1}{4}$$
   $$\langle (\Delta X_1)^2 \rangle = \langle \hat{X}_1^2 \rangle - \langle \hat{X}_1 \rangle^2 = \frac{1}{4}$$
   $$\langle (\Delta X_2)^2 \rangle = \frac{1}{4}$$

### The Universal Shot-Noise Circle
The variances are **strictly independent of amplitude $\alpha$**:

$$\Delta X_1 = \Delta X_2 = \frac{1}{2} \implies \Delta X_1 \Delta X_2 = \frac{1}{4}$$

* In phase space $(X_1, X_2)$, a coherent state is represented by an **uncertainty circle of fixed radius $r = 1/2$** centered at $(\mathrm{Re}(\alpha), \mathrm{Im}(\alpha))$.
* A coherent state has the **exact same quantum noise profile as the vacuum $|0\rangle$**, simply displaced away from the origin by vector $\alpha$!


In [ ]:
def plot_phase_space_circles():
    alphas = [0.0, 2.0, 2.5 + 2.5j, -3.0 + 1.5j]
    labels = [r"Vacuum $|0\rangle$", r"$\alpha = 2.0$", r"$\alpha = 2.5 + 2.5i$", r"$\alpha = -3.0 + 1.5i$"]
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    
    fig, ax = plt.subplots(figsize=(7, 7))
    theta = np.linspace(0, 2*np.pi, 200)
    radius_noise = 0.5  # Delta X1 = Delta X2 = 1/2
    
    for i, a_val in enumerate(alphas):
        x0 = np.real(a_val)
        y0 = np.imag(a_val)
        
        # Uncertainty circle boundary
        x_circ = x0 + radius_noise * np.cos(theta)
        y_circ = y0 + radius_noise * np.sin(theta)
        
        ax.plot(x_circ, y_circ, color=colors[i], lw=2.0, label=labels[i])
        ax.fill(x_circ, y_circ, color=colors[i], alpha=0.15)
        ax.plot([0, x0], [0, y0], color=colors[i], ls=":", lw=1.2)
        ax.scatter([x0], [y0], color=colors[i], s=50)
        
    ax.axhline(0, color="gray", lw=1)
    ax.axvline(0, color="gray", lw=1)
    ax.set_aspect("equal")
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_title(r"Phase Space Quadrature Representation $(X_1, X_2)$")
    ax.set_xlabel(r"In-Phase Quadrature $X_1 = \mathrm{Re}(\alpha)$")
    ax.set_ylabel(r"Out-of-Phase Quadrature $X_2 = \mathrm{Im}(\alpha)$")
    ax.legend(loc="upper left")
    
    plt.tight_layout()
    plt.show()

plot_phase_space_circles()


---
## 5. Non-Orthogonality & Overcompleteness

Unlike Fock states which are strictly orthogonal ($\langle m | n \rangle = \delta_{mn}$), two distinct coherent states $|\alpha\rangle$ and $|\beta\rangle$ are **never strictly orthogonal**!

### Overlap Integral
Evaluating the inner product using the Fock expansion:

$$\langle \beta | \alpha \rangle = e^{-\frac{1}{2}(|\alpha|^2 + |\beta|^2)} \sum_{n=0}^{\infty} \frac{(\beta^* \alpha)^n}{n!} = \exp\left( -\frac{1}{2}|\alpha|^2 - \frac{1}{2}|\beta|^2 + \beta^* \alpha \right)$$

Taking the modulus squared gives the transition probability:

$$|\langle \beta | \alpha \rangle|^2 = \exp\left( -|\alpha - \beta|^2 \right)$$

* If $\alpha = \beta$: $|\langle \alpha | \alpha \rangle|^2 = 1.0$.
* If $|\alpha - \beta| \gg 1$: The overlap decays exponentially toward zero, rendering well-separated states asymptotically orthogonal.
* For opposite-phase states $|\alpha\rangle$ and $|-\alpha\rangle$:
  $$|\langle -\alpha | \alpha \rangle|^2 = e^{-4|\alpha|^2}$$
  Even for modest amplitude $|\alpha| = 2$ ($\bar{n} = 4$), the overlap is $e^{-16} \approx 1.1 \times 10^{-7}$ (practically zero!).

### Overcompleteness of the Coherent Basis
Because there are continuously infinite coherent states $|\alpha\rangle$ across the 2D complex plane $\mathbb{C}$, the basis is **overcomplete**. The resolution of identity is:

$$\frac{1}{\pi} \int_{\mathbb{C}} |\alpha\rangle\langle\alpha|\,d^2\alpha = \mathbb{I}$$

where $d^2\alpha = d(\mathrm{Re}\alpha)\,d(\mathrm{Im}\alpha)$.


In [ ]:
def test_coherent_overlap():
    alpha_0 = 1.5
    sep_range = np.linspace(0, 4.0, 100)
    overlaps_exact = np.exp(-sep_range**2)
    
    overlaps_numeric = []
    N_dim = 40
    
    for s in sep_range:
        beta = alpha_0 + s
        if has_qutip:
            psi_a = coherent(N_dim, alpha_0)
            psi_b = coherent(N_dim, beta)
            ov = abs((psi_b.dag() * psi_a).tr())**2
        else:
            psi_a = construct_coherent_state(N_dim, alpha_0)
            psi_b = construct_coherent_state(N_dim, beta)
            ov = abs(np.vdot(psi_b, psi_a))**2
        overlaps_numeric.append(ov)
        
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(sep_range, overlaps_numeric, 'bo', markersize=4, label=r"Numerical Overlap $|\langle \beta | \alpha \rangle|^2$")
    ax.plot(sep_range, overlaps_exact, 'r--', lw=2, label=r"Analytical $\exp(-|\alpha - \beta|^2)$")
    ax.set_title(r"Coherent State Non-Orthogonality vs. Separation $|\alpha - \beta|$")
    ax.set_xlabel(r"Separation Distance $|\alpha - \beta|$")
    ax.set_ylabel(r"Overlap Fidelity $|\langle \beta | \alpha \rangle|^2$")
    ax.set_yscale("log")
    ax.set_ylim(1e-7, 1.5)
    ax.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

test_coherent_overlap()


---
## 6. Interactive Playground: Coherent State Phase Space Simulator

Use the interactive sliders below to dynamically adjust the **amplitude $|\alpha|$** and **optical phase $\theta$**.

Observe how:
1. The phase-space circle rotates around the origin at radius $|\alpha|$.
2. The Poissonian photon number distribution shifts its mean to $\bar{n} = |\alpha|^2$ while remaining completely insensitive to phase $\theta$!


In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    has_widgets = True
except ImportError:
    has_widgets = False

def interactive_coherent_explorer(mag=2.0, phase_factor=0.25):
    theta = phase_factor * np.pi
    alpha = mag * np.exp(1j * theta)
    n_bar = mag**2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 1. Phase space plot
    theta_circ = np.linspace(0, 2*np.pi, 200)
    x0 = np.real(alpha)
    y0 = np.imag(alpha)
    
    # Orbit trajectory
    ax1.plot(mag * np.cos(theta_circ), mag * np.sin(theta_circ), 'k:', lw=1, alpha=0.5, label=rf"Orbit $|\alpha| = {mag:.2f}$")
    
    # Noise circle (radius 0.5)
    ax1.plot(x0 + 0.5 * np.cos(theta_circ), y0 + 0.5 * np.sin(theta_circ), color="#d62728", lw=2)
    ax1.fill(x0 + 0.5 * np.cos(theta_circ), y0 + 0.5 * np.sin(theta_circ), color="#d62728", alpha=0.2)
    ax1.plot([0, x0], [0, y0], color="#d62728", lw=1.5)
    ax1.scatter([x0], [y0], color="#d62728", s=60, label=rf"$\alpha = {x0:.2f} + {y0:.2f}i$")
    
    ax1.axhline(0, color="gray", lw=1)
    ax1.axvline(0, color="gray", lw=1)
    ax1.set_aspect("equal")
    ax1.set_xlim(-4.5, 4.5)
    ax1.set_ylim(-4.5, 4.5)
    ax1.set_title(r"Phase Space Quadrature Position $(X_1, X_2)$")
    ax1.set_xlabel(r"$X_1 = \mathrm{Re}(\alpha)$")
    ax1.set_ylabel(r"$X_2 = \mathrm{Im}(\alpha)$")
    ax1.legend(loc="upper left")
    
    # 2. Photon Number Statistics
    fock_k = np.arange(25)
    poiss = [np.exp(-n_bar) * (n_bar**k) / float(math.factorial(k)) for k in fock_k]
    
    ax2.bar(fock_k, poiss, color="#1f77b4", width=0.6, edgecolor="black", alpha=0.75)
    ax2.axvline(n_bar, color="#d62728", ls="--", lw=2, label=rf"Mean $\bar{{n}} = |\alpha|^2 = {n_bar:.2f}$")
    ax2.set_title("Photon Number Statistics (Phase-Independent)")
    ax2.set_xlabel("Photon Number $n$")
    ax2.set_ylabel("Probability $P(n)$")
    ax2.set_xlim(-0.5, 24)
    ax2.set_ylim(0, max(0.4, max(poiss)*1.2))
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

if has_widgets:
    interact(
        interactive_coherent_explorer,
        mag=FloatSlider(min=0.0, max=3.5, step=0.1, value=2.0, description="Mag |α|:"),
        phase_factor=FloatSlider(min=0.0, max=2.0, step=0.05, value=0.25, description="Phase θ/π:")
    )
else:
    print("Interactive sliders active in Jupyter/Colab. Displaying baseline:")
    interactive_coherent_explorer(mag=2.0, phase_factor=0.25)


---
## 7. Hands-On Exercises & Guided Challenges

### Exercise 1: Time Evolution of a Coherent State
Under the free harmonic Hamiltonian $\hat{H} = \hbar\omega \hat{a}^\dagger\hat{a}$:
1. Prove analytically that a coherent state $|\alpha_0\rangle$ remains a coherent state for all time, with its complex amplitude rotating as:
   $$|\alpha(t)\rangle = \exp\left(-\frac{i\hat{H}t}{\hbar}\right)|\alpha_0\rangle = |\alpha_0 e^{-i\omega t}\rangle$$
2. Explain why the mean quadrature vector $\vec{X}(t) = (\langle X_1(t)\rangle, \langle X_2(t)\rangle)$ traces a circle clockwise in phase space with period $T = 2\pi/\omega$.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Analytical proof:**
Expanding $|\alpha_0\rangle$ in the Fock basis:
$$|\alpha(t)\rangle = e^{-i\omega\hat{n}t}|\alpha_0\rangle = e^{-|\alpha_0|^2/2} \sum_{n=0}^\infty \frac{\alpha_0^n}{\sqrt{n!}} e^{-in\omega t}|n\rangle$$
Grouping terms:
$$\alpha_0^n e^{-in\omega t} = (\alpha_0 e^{-i\omega t})^n$$
Therefore:
$$|\alpha(t)\rangle = e^{-|\alpha_0|^2/2} \sum_{n=0}^\infty \frac{(\alpha_0 e^{-i\omega t})^n}{\sqrt{n!}} |n\rangle = |\alpha_0 e^{-i\omega t}\rangle$$
Since $|\alpha_0 e^{-i\omega t}|^2 = |\alpha_0|^2$, the mean photon number is strictly conserved, and the state rotates steadily clockwise at classical angular frequency $\omega$.
</details>

---

### Exercise 2: State Overlap & The Cat State Generator
Consider two coherent states with equal magnitude but opposite phase: $|\alpha\rangle$ and $|-\alpha\rangle$.
1. Compute the overlap fidelity $F = |\langle -\alpha | \alpha \rangle|^2$ for $\alpha = 2.0$.
2. In Module 2, we will construct the **Schrödinger Cat State**:
   $$|\psi_{\text{cat}}\rangle = \frac{1}{\sqrt{\mathcal{N}}} \left( |\alpha\rangle + |-\alpha\rangle \right)$$
   Show that the normalization constant is $\mathcal{N} = 2(1 + e^{-2|\alpha|^2})$.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Solution:**
1. $|\langle -\alpha | \alpha \rangle|^2 = \exp(-|\alpha - (-\alpha)|^2) = \exp(-|2\alpha|^2) = \exp(-4|\alpha|^2)$.
   For $\alpha = 2.0$, $|\langle -2 | 2 \rangle|^2 = e^{-16} \approx 1.125 \times 10^{-7}$.
2. Normalization:
   $$\langle\psi_{\text{cat}}|\psi_{\text{cat}}\rangle = \frac{1}{\mathcal{N}} \left( \langle\alpha|\alpha\rangle + \langle-\alpha|-\alpha\rangle + \langle\alpha|-\alpha\rangle + \langle-\alpha|\alpha\rangle \right)$$
   Using $\langle\alpha|\alpha\rangle = 1$ and $\langle-\alpha|\alpha\rangle = e^{-2|\alpha|^2}$:
   $$\langle\psi_{\text{cat}}|\psi_{\text{cat}}\rangle = \frac{1}{\mathcal{N}} \left( 1 + 1 + 2e^{-2|\alpha|^2} \right) = \frac{2(1 + e^{-2|\alpha|^2})}{\mathcal{N}} \equiv 1$$
   $$\implies \mathcal{N} = 2(1 + e^{-2|\alpha|^2})$$
</details>


---
## 8. Summary & What's Next

### Key Takeaways
1. **Three Equivalent Formulations:** Coherent states $|\alpha\rangle$ are right-eigenstates of annihilation operator $\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$, displaced vacuum states $\hat{D}(\alpha)|0\rangle$, and minimum uncertainty states ($\Delta X_1 \Delta X_2 = 1/4$).
2. **Poissonian Counting Statistics:** Photon distribution $P(n) = e^{-\bar{n}}\frac{\bar{n}^n}{n!}$ exhibits variance equal to the mean ($\langle (\Delta n)^2 \rangle = \bar{n}$), benchmarked by Fano factor $F = 1$.
3. **Phase Space Geometry:** A coherent state has the identical isotropic shot-noise circle of radius $1/2$ as the vacuum, displaced to $(\mathrm{Re}(\alpha), \mathrm{Im}(\alpha))$.
4. **Non-Orthogonality:** Overlap $|\langle \beta | \alpha \rangle|^2 = e^{-|\alpha-\beta|^2}$ drops exponentially with distance in phase space, forming an overcomplete basis.

---

### What's Next?
In **Notebook 1.3 (`03_squeezed_states_and_quadrature_noise.ipynb`)**, we will:
* Implement the **squeeze operator** $\hat{S}(\xi) = \exp\left(\frac{1}{2}(\xi^* \hat{a}^2 - \xi \hat{a}^{\dagger 2})\right)$.
* Generate **Squeezed Vacuum States** and compress noise in one quadrature below the vacuum shot-noise limit ($\Delta X_1 < 1/2$).
* Discover why squeezed vacuum contains exclusively **even Fock states** ($|0\rangle, |2\rangle, |4\rangle, \dots$) generated in photon pairs!

---
### References & Recommended Reading
* **Roy J. Glauber**, *Coherent and Incoherent States of the Radiation Field*, Phys. Rev. 131, 2766 (1963).
* **Christopher Gerry & Peter Knight**, *Introductory Quantum Optics*, Chapter 3: Coherent States.
* **Marlan O. Scully & M. Suhail Zubairy**, *Quantum Optics*, Chapter 2: Coherent States and Phase Space.
